# Figure S1 — cohort overlap Venn diagrams

Two three-set Venn diagrams showing modality overlap across the study cohorts:

* **A. In-vivo cohort** — Cranial ultrasound &times; Brain MRI &times; Neurodevelopmental follow-up
 (surviving preterm infants; ND restricted to the analytic EP/VP window).
* **B. Post-mortem cohort** — Deep-learning neuropathology &times; Spatial transcriptomics &times; *In situ* validation.

*Only the Venn diagrams are generated here; the accompanying CONSORT-style flow diagram is a designed asset.*

**Reproducibility.** Data are read from the repo-local `Supplementary_Datasets/` folder via a relative path
(`find_dataset` walks up to a `Supplementary_Datasets/` directory), so no machine- or user-specific path is embedded.
Colours use the Okabe-Ito colour-blind-safe palette.

In [1]:
import warnings; warnings.filterwarnings("ignore")  # clean render: hide non-fatal warnings
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import openpyxl
from venn import venn as venn_ellipse

INK = "#1A202C"

def find_dataset(fname, start="."):
    """Return a RELATIVE path to `fname` inside a `Supplementary_Datasets/` folder found by walking up."""
    d = os.path.abspath(start)
    rel = ""
    while True:
        cand = os.path.join(d, "Supplementary_Datasets", fname)
        if os.path.exists(cand):
            return os.path.join(rel, "Supplementary_Datasets", fname) if rel else os.path.join("Supplementary_Datasets", fname)
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(f"{fname} not found under any Supplementary_Datasets/ folder above {start}")
        d = parent
        rel = os.path.join("..", rel) if rel else ".."

def is_yes(v):
    return str(v).strip().lower() in ("1", "yes", "y", "true", "x")

def read_sheet(path, sheet):
    ws = openpyxl.load_workbook(path, read_only=True, data_only=True)[sheet]
    rows = list(ws.iter_rows(values_only=True))
    header = list(rows[0])
    return header, rows[1:]

In [2]:
def draw_venn(sets, order, labels, palette, out_stem, label_pos):
    """Draw a 3-set proportional-ellipse Venn; region counts inside, black labels beside each circle.

    label_pos: list of (x, y, ha, va) per set, placing each modality label next to its own circle."""
    fig = plt.figure(figsize=(7.0, 7.4))
    ax = fig.add_axes([0.02, 0.06, 0.96, 0.90]); ax.set_aspect("equal")
    venn_ellipse({s: sets[s] for s in order}, ax=ax, fmt="{size}",
                 fontsize=17, cmap=palette[:3], legend_loc="upper left")
    for t in ax.texts:                      # region-count labels
        t.set_fontweight("normal"); t.set_fontsize(17)
        if t.get_text() == "0":
            t.set_text("")                  # hide empty regions
    lg = ax.get_legend()
    if lg: lg.remove()                      # replace default legend with placed labels
    ax.set_xlim(-0.22, 1.22); ax.set_ylim(-0.20, 1.22)
    for i, s in enumerate(order):
        x, y, ha, va = label_pos[i]
        ax.text(x, y, labels[i], ha=ha, va=va,
                fontsize=13, color=INK, fontweight="bold")
    fig.savefig(out_stem + ".pdf")
    plt.show()
    plt.close(fig)
    return {s: len(sets[s]) for s in order}

## A. In-vivo cohort

Membership from the study's de-identified cohort key (`Data_S1_InVivo_Cohort.xlsx`, `InVivo_Cohort` sheet): US / MRI as flagged; neurodevelopmental follow-up counted for the **analytic** window only (ND recorded **and** GA group EP or VP), matching the manuscript's ND-analytic definition.

In [ ]:
iv_path = find_dataset("Data_S1_InVivo_Cohort.xlsx")
header, rows = read_sheet(iv_path, "InVivo_Cohort")
col = {c: header.index(c) for c in ("Study_ID", "US", "MRI", "ND", "GA_Group")}

US, MRI, ND = set(), set(), set()
for r in rows:
    sid = r[col["Study_ID"]]
    if sid is None:
        continue
    if is_yes(r[col["US"]]):  US.add(sid)
    if is_yes(r[col["MRI"]]): MRI.add(sid)
    if is_yes(r[col["ND"]]) and str(r[col["GA_Group"]]).strip().upper() in ("EP", "VP"):
        ND.add(sid)

iv_sets = {"US": US, "MRI": MRI, "ND": ND}
OKABE_IV = ["#0072B2", "#E69F00", "#009E73"]   # blue / orange / green
sizes_iv = draw_venn(
    iv_sets, ["US", "MRI", "ND"],
    ["Cranial ultrasound", "Brain MRI", "Neurodevelopmental\nfollow-up"],
    OKABE_IV, "Figure_S1A_InVivo_Venn",
    [(0.06, 0.99, "left", "center"),      # Cranial ultrasound — beside upper-left circle
     (0.94, 0.99, "right", "center"),     # Brain MRI — beside upper-right circle
     (0.60, -0.05, "center", "center")])  # ND follow-up — below bottom circle, offset right
print("In-vivo sizes:", sizes_iv, " | total (any):", len(US | MRI | ND))

## B. Post-mortem cohort

Membership from the study's de-identified cohort key (`Data_S2_Postmortem_Histology.xlsx`, `Postmortem_Cohort` sheet): `Histology` (deep-learning neuropathology), `Spatial` (spatial transcriptomics), and *In situ* validation = subjects assayed by **RNAScope** *or* immunofluorescence (**VGLUT1** or **VGLUT2**).

In [ ]:
pm_path = find_dataset("Data_S2_Postmortem_Histology.xlsx")
header, rows = read_sheet(pm_path, "Postmortem_Cohort")
col = {c: header.index(c) for c in ("Paper_ID", "Histology", "Spatial", "RNAScope", "IF_VGLUT1", "IF_VGLUT2")}

H, S, I = set(), set(), set()
for r in rows:
    pid = r[col["Paper_ID"]]
    if pid is None:
        continue
    if is_yes(r[col["Histology"]]): H.add(pid)
    if is_yes(r[col["Spatial"]]):   S.add(pid)
    # In situ validation = union of subjects assayed by RNAScope OR immunofluorescence (VGLUT1 or VGLUT2)
    if is_yes(r[col["RNAScope"]]) or is_yes(r[col["IF_VGLUT1"]]) or is_yes(r[col["IF_VGLUT2"]]):
        I.add(pid)

pm_sets = {"H": H, "S": S, "I": I}
OKABE_PM = ["#CC79A7", "#D55E00", "#56B4E9"]    # magenta / vermillion / sky-blue
sizes_pm = draw_venn(
    pm_sets, ["H", "S", "I"],
    ["Deep-learning\nneuropathology", "Spatial\ntranscriptomics", "In situ validation"],
    OKABE_PM, "Figure_S1B_Postmortem_Venn",
    [(0.06, 0.99, "left", "center"),      # Deep-learning neuropathology — upper-left circle
     (0.94, 0.99, "right", "center"),     # Spatial transcriptomics — upper-right circle
     (0.50, -0.03, "center", "center")])  # In situ validation — below bottom circle
print("Post-mortem sizes:", sizes_pm, " | total (any):", len(H | S | I))